In [7]:
# Feature Engineering

This notebook prepares transaction and budget data for financial analysis and root-cause detection.

The feature engineering process includes:

- Loading validated datasets
- Creating time-based features
- Aggregating transaction-level data
- Joining transaction metrics with budget data
- Calculating budget utilization and variance metrics
- Preparing features required for downstream analysis and anomaly detection

SyntaxError: invalid syntax (1633888427.py, line 3)

In [8]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [13]:
transactions_df = pd.read_csv("../dataset/transactions.csv")
monthly_budgets_df = pd.read_csv("../dataset/monthly_budgets.csv")

print("Transactions shape:", transactions_df.shape)
print("Monthly budgets shape:", monthly_budgets_df.shape)

print("\nTransactions columns:")
print(transactions_df.columns.tolist())

print("\nMonthly budget columns:")
print(monthly_budgets_df.columns.tolist())

Transactions shape: (60000, 10)
Monthly budgets shape: (504, 5)

Transactions columns:
['transaction_id', 'date', 'department', 'category', 'vendor', 'region', 'transaction_type', 'amount', 'payment_method', 'description']

Monthly budget columns:
['month', 'department', 'category', 'actual_expense', 'budget']


In [14]:
transactions_df["date"] = pd.to_datetime(transactions_df["date"])

print("Transaction data types:")
print(transactions_df.dtypes)

print("\nTransaction date range:")
print(transactions_df["date"].min(), "to", transactions_df["date"].max())

print("\nMonthly budget data types:")
print(monthly_budgets_df.dtypes)

print("\nSample transactions:")
display(transactions_df.head())

Transaction data types:
transaction_id              object
date                datetime64[ns]
department                  object
category                    object
vendor                      object
region                      object
transaction_type            object
amount                     float64
payment_method              object
description                 object
dtype: object

Transaction date range:
2025-01-01 00:00:00 to 2026-06-29 00:00:00

Monthly budget data types:
month              object
department         object
category           object
actual_expense    float64
budget            float64
dtype: object

Sample transactions:


,transaction_id,date,department,category,vendor,region,transaction_type,amount,payment_method,description
0,TXN100000,2025-01-01,Sales,Commissions,SalesPartners,West,Expense,22477.74,Credit Card,Commissions transaction for Sales
1,TXN100001,2025-01-01,Customer Support,Service Revenue,Customer_E,West,Revenue,17920.46,Bank Transfer,Service Revenue transaction for Customer Support
2,TXN100002,2025-01-01,Finance,Subscription,Customer_C,West,Revenue,14017.17,Debit Card,Subscription transaction for Finance
3,TXN100003,2025-01-01,Sales,Commissions,SalesPartners,South,Expense,6900.02,Bank Transfer,Commissions transaction for Sales
4,TXN100004,2025-01-01,IT,Cloud Services,DataCloud,South,Expense,10066.59,Debit Card,Cloud Services transaction for IT


In [15]:
# Create monthly period
transactions_df["period"] = transactions_df["date"].dt.to_period("M").astype(str)

# Aggregate transaction-level data
monthly_transactions = (
    transactions_df
    .groupby(["period", "department", "category"], as_index=False)
    .agg(
        transaction_count=("transaction_id", "count"),
        total_amount=("amount", "sum")
    )
)

print("Monthly transaction shape:", monthly_transactions.shape)

print("\nMonthly transaction columns:")
print(monthly_transactions.columns.tolist())

display(monthly_transactions.head(10))

Monthly transaction shape: (1008, 5)

Monthly transaction columns:
['period', 'department', 'category', 'transaction_count', 'total_amount']


,period,department,category,transaction_count,total_amount
0,2025-01,Customer Support,Outsourcing,79,906916.46
1,2025-01,Customer Support,Product Sales,56,1273072.74
2,2025-01,Customer Support,Service Revenue,50,1125824.05
3,2025-01,Customer Support,Software,82,724787.64
4,2025-01,Customer Support,Subscription,60,1456901.64
5,2025-01,Customer Support,Support Tools,69,658831.34
6,2025-01,Customer Support,Training,77,483785.12
7,2025-01,Customer Support,Transaction Fees,31,721542.13
8,2025-01,Finance,Audit,49,276168.70
9,2025-01,Finance,Bank Charges,62,278818.88


In [16]:
# Convert budget month to the same period format
monthly_budgets_df["period"] = pd.to_datetime(
    monthly_budgets_df["month"]
).dt.to_period("M").astype(str)

print("Budget periods:")
print(sorted(monthly_budgets_df["period"].unique()))

print("\nBudget columns:")
print(monthly_budgets_df.columns.tolist())

display(monthly_budgets_df.head())

Budget periods:
['2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']

Budget columns:
['month', 'department', 'category', 'actual_expense', 'budget', 'period']


,month,department,category,actual_expense,budget,period
0,2025-01,Customer Support,Outsourcing,906916.46,913791.70,2025-01
1,2025-01,Customer Support,Software,724787.64,737863.28,2025-01
2,2025-01,Customer Support,Support Tools,658831.34,640835.18,2025-01
3,2025-01,Customer Support,Training,483785.12,477200.44,2025-01
4,2025-01,Finance,Audit,276168.70,270906.81,2025-01


In [17]:
# Merge monthly transactions with budget data
feature_df = monthly_transactions.merge(
    monthly_budgets_df[
        ["period", "department", "category", "budget"]
    ],
    on=["period", "department", "category"],
    how="left"
)

print("Feature dataset shape:", feature_df.shape)

print("\nMissing budget values after merge:")
print(feature_df["budget"].isna().sum())

print("\nFeature dataset columns:")
print(feature_df.columns.tolist())

display(feature_df.head(10))

Feature dataset shape: (1008, 6)

Missing budget values after merge:
504

Feature dataset columns:
['period', 'department', 'category', 'transaction_count', 'total_amount', 'budget']


,period,department,category,transaction_count,total_amount,budget
0,2025-01,Customer Support,Outsourcing,79,906916.46,913791.70
1,2025-01,Customer Support,Product Sales,56,1273072.74,NaN
2,2025-01,Customer Support,Service Revenue,50,1125824.05,NaN
3,2025-01,Customer Support,Software,82,724787.64,737863.28
4,2025-01,Customer Support,Subscription,60,1456901.64,NaN
5,2025-01,Customer Support,Support Tools,69,658831.34,640835.18
6,2025-01,Customer Support,Training,77,483785.12,477200.44
7,2025-01,Customer Support,Transaction Fees,31,721542.13,NaN
8,2025-01,Finance,Audit,49,276168.70,270906.81
9,2025-01,Finance,Bank Charges,62,278818.88,258334.99


In [18]:
# Check which department-category combinations have missing budgets
missing_budget_df = feature_df[feature_df["budget"].isna()].copy()

print("Rows without budget:", len(missing_budget_df))

print("\nMissing budget by department:")
print(missing_budget_df["department"].value_counts())

print("\nMissing budget by category:")
print(missing_budget_df["category"].value_counts())

print("\nMissing department-category combinations:")
display(
    missing_budget_df[
        ["department", "category"]
    ].drop_duplicates().sort_values(
        ["department", "category"]
    )
)

Rows without budget: 504

Missing budget by department:
department
Customer Support    72
Finance             72
HR                  72
IT                  72
Marketing           72
Operations          72
Sales               72
Name: count, dtype: int64

Missing budget by category:
category
Product Sales       126
Service Revenue     126
Subscription        126
Transaction Fees    126
Name: count, dtype: int64

Missing department-category combinations:


,department,category
1,Customer Support,Product Sales
2,Customer Support,Service Revenue
4,Customer Support,Subscription
7,Customer Support,Transaction Fees
10,Finance,Product Sales
12,Finance,Service Revenue
14,Finance,Subscription
15,Finance,Transaction Fees
17,HR,Product Sales
19,HR,Service Revenue


In [19]:
print("Missing budget by category:")
print(missing_budget_df["category"].value_counts())

print("\nMissing department-category combinations:")
display(
    missing_budget_df[
        ["department", "category"]
    ]
    .drop_duplicates()
    .sort_values(["department", "category"])
)

Missing budget by category:
category
Product Sales       126
Service Revenue     126
Subscription        126
Transaction Fees    126
Name: count, dtype: int64

Missing department-category combinations:


,department,category
1,Customer Support,Product Sales
2,Customer Support,Service Revenue
4,Customer Support,Subscription
7,Customer Support,Transaction Fees
10,Finance,Product Sales
12,Finance,Service Revenue
14,Finance,Subscription
15,Finance,Transaction Fees
17,HR,Product Sales
19,HR,Service Revenue


In [20]:
# Calculate budget variance
feature_df["variance"] = (
    feature_df["total_amount"] - feature_df["budget"]
)

# Calculate budget utilization percentage
feature_df["budget_utilization_pct"] = (
    feature_df["total_amount"] / feature_df["budget"]
) * 100

# Calculate variance percentage
feature_df["variance_pct"] = (
    feature_df["variance"] / feature_df["budget"]
) * 100

print("Feature engineering completed.")

display(feature_df.head(10))

Feature engineering completed.


,period,department,category,transaction_count,total_amount,budget,variance,budget_utilization_pct,variance_pct
0,2025-01,Customer Support,Outsourcing,79,906916.46,913791.70,-6875.24,99.247614,-0.752386
1,2025-01,Customer Support,Product Sales,56,1273072.74,NaN,NaN,NaN,NaN
2,2025-01,Customer Support,Service Revenue,50,1125824.05,NaN,NaN,NaN,NaN
3,2025-01,Customer Support,Software,82,724787.64,737863.28,-13075.64,98.227905,-1.772095
4,2025-01,Customer Support,Subscription,60,1456901.64,NaN,NaN,NaN,NaN
5,2025-01,Customer Support,Support Tools,69,658831.34,640835.18,17996.16,102.808235,2.808235
6,2025-01,Customer Support,Training,77,483785.12,477200.44,6584.68,101.379856,1.379856
7,2025-01,Customer Support,Transaction Fees,31,721542.13,NaN,NaN,NaN,NaN
8,2025-01,Finance,Audit,49,276168.70,270906.81,5261.89,101.942325,1.942325
9,2025-01,Finance,Bank Charges,62,278818.88,258334.99,20483.89,107.929197,7.929197


In [21]:
# Analyze missing-budget rows
missing_budget_df = feature_df[feature_df["budget"].isna()].copy()

print("Rows without budget:", len(missing_budget_df))

print("\nMissing budget by department:")
print(missing_budget_df["department"].value_counts())

print("\nMissing budget by category:")
print(missing_budget_df["category"].value_counts())

print("\nMissing department-category combinations:")
display(
    missing_budget_df[
        ["department", "category"]
    ]
    .drop_duplicates()
    .sort_values(["department", "category"])
)

Rows without budget: 504

Missing budget by department:
department
Customer Support    72
Finance             72
HR                  72
IT                  72
Marketing           72
Operations          72
Sales               72
Name: count, dtype: int64

Missing budget by category:
category
Product Sales       126
Service Revenue     126
Subscription        126
Transaction Fees    126
Name: count, dtype: int64

Missing department-category combinations:


,department,category
1,Customer Support,Product Sales
2,Customer Support,Service Revenue
4,Customer Support,Subscription
7,Customer Support,Transaction Fees
10,Finance,Product Sales
12,Finance,Service Revenue
14,Finance,Subscription
15,Finance,Transaction Fees
17,HR,Product Sales
19,HR,Service Revenue


In [22]:
feature_df["budget_available"] = feature_df["budget"].notna().astype(int)

print("Budget availability:")
print(feature_df["budget_available"].value_counts())

display(
    feature_df[
        ["period", "department", "category", "budget", "budget_available"]
    ].head(10)
)

Budget availability:
budget_available
1    504
0    504
Name: count, dtype: int64


,period,department,category,budget,budget_available
0,2025-01,Customer Support,Outsourcing,913791.70,1
1,2025-01,Customer Support,Product Sales,NaN,0
2,2025-01,Customer Support,Service Revenue,NaN,0
3,2025-01,Customer Support,Software,737863.28,1
4,2025-01,Customer Support,Subscription,NaN,0
5,2025-01,Customer Support,Support Tools,640835.18,1
6,2025-01,Customer Support,Training,477200.44,1
7,2025-01,Customer Support,Transaction Fees,NaN,0
8,2025-01,Finance,Audit,270906.81,1
9,2025-01,Finance,Bank Charges,258334.99,1


In [23]:

# Calculate budget variance
feature_df["variance"] = (
    feature_df["total_amount"] - feature_df["budget"]
)

# Calculate budget utilization percentage
feature_df["budget_utilization_pct"] = (
    feature_df["total_amount"] / feature_df["budget"]
) * 100

# Calculate variance percentage
feature_df["variance_pct"] = (
    feature_df["variance"] / feature_df["budget"]
) * 100

print("Feature engineering completed.")

display(feature_df.head(10))


Feature engineering completed.


,period,department,category,transaction_count,total_amount,budget,variance,budget_utilization_pct,variance_pct,budget_available
0,2025-01,Customer Support,Outsourcing,79,906916.46,913791.70,-6875.24,99.247614,-0.752386,1
1,2025-01,Customer Support,Product Sales,56,1273072.74,NaN,NaN,NaN,NaN,0
2,2025-01,Customer Support,Service Revenue,50,1125824.05,NaN,NaN,NaN,NaN,0
3,2025-01,Customer Support,Software,82,724787.64,737863.28,-13075.64,98.227905,-1.772095,1
4,2025-01,Customer Support,Subscription,60,1456901.64,NaN,NaN,NaN,NaN,0
5,2025-01,Customer Support,Support Tools,69,658831.34,640835.18,17996.16,102.808235,2.808235,1
6,2025-01,Customer Support,Training,77,483785.12,477200.44,6584.68,101.379856,1.379856,1
7,2025-01,Customer Support,Transaction Fees,31,721542.13,NaN,NaN,NaN,NaN,0
8,2025-01,Finance,Audit,49,276168.70,270906.81,5261.89,101.942325,1.942325,1
9,2025-01,Finance,Bank Charges,62,278818.88,258334.99,20483.89,107.929197,7.929197,1


In [24]:
# Step 8 — Create spending intensity feature

feature_df["avg_transaction_value"] = (
    feature_df["total_amount"] /
    feature_df["transaction_count"]
)

print("Average transaction value created.")

display(
    feature_df[
        [
            "period",
            "department",
            "category",
            "transaction_count",
            "total_amount",
            "avg_transaction_value"
        ]
    ].head(10)
)

Average transaction value created.


,period,department,category,transaction_count,total_amount,avg_transaction_value
0,2025-01,Customer Support,Outsourcing,79,906916.46,11479.955190
1,2025-01,Customer Support,Product Sales,56,1273072.74,22733.441786
2,2025-01,Customer Support,Service Revenue,50,1125824.05,22516.481000
3,2025-01,Customer Support,Software,82,724787.64,8838.873659
4,2025-01,Customer Support,Subscription,60,1456901.64,24281.694000
5,2025-01,Customer Support,Support Tools,69,658831.34,9548.280290
6,2025-01,Customer Support,Training,77,483785.12,6282.923636
7,2025-01,Customer Support,Transaction Fees,31,721542.13,23275.552581
8,2025-01,Finance,Audit,49,276168.70,5636.095918
9,2025-01,Finance,Bank Charges,62,278818.88,4497.078710


In [25]:
# Step 9 — Calculate spending deviation from category average

category_avg = (
    feature_df
    .groupby(["department", "category"])["total_amount"]
    .transform("mean")
)

feature_df["spending_deviation_pct"] = (
    (feature_df["total_amount"] - category_avg)
    / category_avg
) * 100

print("Spending deviation feature created.")

display(
    feature_df[
        [
            "period",
            "department",
            "category",
            "total_amount",
            "spending_deviation_pct"
        ]
    ].head(10)
)

Spending deviation feature created.


,period,department,category,total_amount,spending_deviation_pct
0,2025-01,Customer Support,Outsourcing,906916.46,0.903640
1,2025-01,Customer Support,Product Sales,1273072.74,9.997398
2,2025-01,Customer Support,Service Revenue,1125824.05,3.065599
3,2025-01,Customer Support,Software,724787.64,2.270978
4,2025-01,Customer Support,Subscription,1456901.64,34.795807
5,2025-01,Customer Support,Support Tools,658831.34,-12.701382
6,2025-01,Customer Support,Training,483785.12,-11.456883
7,2025-01,Customer Support,Transaction Fees,721542.13,-32.509749
8,2025-01,Finance,Audit,276168.70,-23.223205
9,2025-01,Finance,Bank Charges,278818.88,19.364467


In [26]:
# Step 10 — Calculate transaction count deviation

transaction_count_avg = (
    feature_df
    .groupby(["department", "category"])["transaction_count"]
    .transform("mean")
)

feature_df["transaction_count_deviation_pct"] = (
    (feature_df["transaction_count"] - transaction_count_avg)
    / transaction_count_avg
) * 100

print("Transaction count deviation feature created.")

display(
    feature_df[
        [
            "period",
            "department",
            "category",
            "transaction_count",
            "transaction_count_deviation_pct"
        ]
    ].head(10)
)

Transaction count deviation feature created.


,period,department,category,transaction_count,transaction_count_deviation_pct
0,2025-01,Customer Support,Outsourcing,79,0.708215
1,2025-01,Customer Support,Product Sales,56,11.258278
2,2025-01,Customer Support,Service Revenue,50,6.257379
3,2025-01,Customer Support,Software,82,9.414381
4,2025-01,Customer Support,Subscription,60,26.909518
5,2025-01,Customer Support,Support Tools,69,-13.929314
6,2025-01,Customer Support,Training,77,0.946832
7,2025-01,Customer Support,Transaction Fees,31,-35.266821
8,2025-01,Finance,Audit,49,-4.751620
9,2025-01,Finance,Bank Charges,62,20.910076


In [27]:
# Step 11 — Calculate total spending for each period

period_total_spending = (
    feature_df
    .groupby("period")["total_amount"]
    .transform("sum")
)

feature_df["period_total_spending"] = period_total_spending

print("Period total spending feature created.")

display(
    feature_df[
        [
            "period",
            "department",
            "category",
            "total_amount",
            "period_total_spending"
        ]
    ].head(10)
)

Period total spending feature created.


,period,department,category,total_amount,period_total_spending
0,2025-01,Customer Support,Outsourcing,906916.46,53363242.28
1,2025-01,Customer Support,Product Sales,1273072.74,53363242.28
2,2025-01,Customer Support,Service Revenue,1125824.05,53363242.28
3,2025-01,Customer Support,Software,724787.64,53363242.28
4,2025-01,Customer Support,Subscription,1456901.64,53363242.28
5,2025-01,Customer Support,Support Tools,658831.34,53363242.28
6,2025-01,Customer Support,Training,483785.12,53363242.28
7,2025-01,Customer Support,Transaction Fees,721542.13,53363242.28
8,2025-01,Finance,Audit,276168.70,53363242.28
9,2025-01,Finance,Bank Charges,278818.88,53363242.28


In [28]:
# Step 12 — Calculate period-over-period spending change

period_totals = (
    feature_df[["period", "period_total_spending"]]
    .drop_duplicates()
    .sort_values("period")
)

period_totals["period_spending_change_pct"] = (
    period_totals["period_total_spending"]
    .pct_change() * 100
)

feature_df = feature_df.merge(
    period_totals[["period", "period_spending_change_pct"]],
    on="period",
    how="left"
)

print("Period-over-period spending change feature created.")

display(
    feature_df[
        [
            "period",
            "department",
            "category",
            "period_total_spending",
            "period_spending_change_pct"
        ]
    ].head(20)
)

Period-over-period spending change feature created.


,period,department,category,period_total_spending,period_spending_change_pct
0,2025-01,Customer Support,Outsourcing,53363242.28,NaN
1,2025-01,Customer Support,Product Sales,53363242.28,NaN
2,2025-01,Customer Support,Service Revenue,53363242.28,NaN
3,2025-01,Customer Support,Software,53363242.28,NaN
4,2025-01,Customer Support,Subscription,53363242.28,NaN
5,2025-01,Customer Support,Support Tools,53363242.28,NaN
6,2025-01,Customer Support,Training,53363242.28,NaN
7,2025-01,Customer Support,Transaction Fees,53363242.28,NaN
8,2025-01,Finance,Audit,53363242.28,NaN
9,2025-01,Finance,Bank Charges,53363242.28,NaN


In [29]:
print("Period spending change check:")

display(
    feature_df[
        ["period", "period_total_spending", "period_spending_change_pct"]
    ].drop_duplicates()
    .sort_values("period")
    .head(10)
)

Period spending change check:


,period,period_total_spending,period_spending_change_pct
0,2025-01,53363242.28,NaN
56,2025-02,49205537.23,-7.791328
112,2025-03,59577029.34,21.077896
168,2025-04,54481805.98,-8.552329
224,2025-05,58516281.48,7.405179
280,2025-06,51348851.16,-12.248609
336,2025-07,51614641.62,0.517617
392,2025-08,50829734.44,-1.520706
448,2025-09,46872855.13,-7.784576
504,2025-10,48992050.97,4.521158


In [30]:
# Step 13 — Calculate 3-month rolling spending average

period_summary = (
    feature_df[
        ["period", "period_total_spending"]
    ]
    .drop_duplicates()
    .sort_values("period")
    .copy()
)

period_summary["rolling_3m_avg_spending"] = (
    period_summary["period_total_spending"]
    .rolling(window=3, min_periods=1)
    .mean()
)

feature_df = feature_df.merge(
    period_summary[
        ["period", "rolling_3m_avg_spending"]
    ],
    on="period",
    how="left"
)

print("3-month rolling spending average created.")

display(
    feature_df[
        [
            "period",
            "period_total_spending",
            "rolling_3m_avg_spending"
        ]
    ]
    .drop_duplicates()
    .sort_values("period")
    .head(10)
)

3-month rolling spending average created.


,period,period_total_spending,rolling_3m_avg_spending
0,2025-01,53363242.28,5.336324e+07
56,2025-02,49205537.23,5.128439e+07
112,2025-03,59577029.34,5.404860e+07
168,2025-04,54481805.98,5.442146e+07
224,2025-05,58516281.48,5.752504e+07
280,2025-06,51348851.16,5.478231e+07
336,2025-07,51614641.62,5.382659e+07
392,2025-08,50829734.44,5.126441e+07
448,2025-09,46872855.13,4.977241e+07
504,2025-10,48992050.97,4.889821e+07


In [31]:
# Step 14 — Calculate deviation from rolling 3-month baseline

feature_df["rolling_spending_deviation_pct"] = (
    (
        feature_df["period_total_spending"]
        - feature_df["rolling_3m_avg_spending"]
    )
    / feature_df["rolling_3m_avg_spending"]
) * 100

print("Rolling spending deviation feature created.")

display(
    feature_df[
        [
            "period",
            "period_total_spending",
            "rolling_3m_avg_spending",
            "rolling_spending_deviation_pct"
        ]
    ]
    .drop_duplicates()
    .sort_values("period")
    .head(10)
)

Rolling spending deviation feature created.


,period,period_total_spending,rolling_3m_avg_spending,rolling_spending_deviation_pct
0,2025-01,53363242.28,5.336324e+07,0.000000
56,2025-02,49205537.23,5.128439e+07,-4.053578
112,2025-03,59577029.34,5.404860e+07,10.228620
168,2025-04,54481805.98,5.442146e+07,0.110891
224,2025-05,58516281.48,5.752504e+07,1.723150
280,2025-06,51348851.16,5.478231e+07,-6.267464
336,2025-07,51614641.62,5.382659e+07,-4.109400
392,2025-08,50829734.44,5.126441e+07,-0.847907
448,2025-09,46872855.13,4.977241e+07,-5.825628
504,2025-10,48992050.97,4.889821e+07,0.191904


In [32]:
# Step 15 — Save engineered features

feature_df.to_csv(
    "../dataset/feature_dataset.csv",
    index=False
)

print("Feature dataset saved successfully!")
print("Shape:", feature_df.shape)

display(feature_df.head())

Feature dataset saved successfully!
Shape: (1008, 17)


,period,department,category,transaction_count,total_amount,budget,variance,budget_utilization_pct,variance_pct,budget_available,avg_transaction_value,spending_deviation_pct,transaction_count_deviation_pct,period_total_spending,period_spending_change_pct,rolling_3m_avg_spending,rolling_spending_deviation_pct
0,2025-01,Customer Support,Outsourcing,79,906916.46,913791.70,-6875.24,99.247614,-0.752386,1,11479.955190,0.903640,0.708215,53363242.28,NaN,53363242.28,0.0
1,2025-01,Customer Support,Product Sales,56,1273072.74,NaN,NaN,NaN,NaN,0,22733.441786,9.997398,11.258278,53363242.28,NaN,53363242.28,0.0
2,2025-01,Customer Support,Service Revenue,50,1125824.05,NaN,NaN,NaN,NaN,0,22516.481000,3.065599,6.257379,53363242.28,NaN,53363242.28,0.0
3,2025-01,Customer Support,Software,82,724787.64,737863.28,-13075.64,98.227905,-1.772095,1,8838.873659,2.270978,9.414381,53363242.28,NaN,53363242.28,0.0
4,2025-01,Customer Support,Subscription,60,1456901.64,NaN,NaN,NaN,NaN,0,24281.694000,34.795807,26.909518,53363242.28,NaN,53363242.28,0.0
